<a href="https://colab.research.google.com/github/hsrkl/techrush26/blob/notebooks/FF_detection_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE
from datasets import load_dataset

In [ ]:
path = kagglehub.dataset_download("ealaxi/paysim1")
df = pd.read_csv(os.path.join(path, "PS_20174392719_1491204439457_log.csv"))
df.drop(columns=['isFlaggedFraud'], inplace=True)

In [ ]:
df.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.0,0.00,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.0,0.00,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.0,0.00,1
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.0,0.00,1
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.0,0.00,0
5,1,PAYMENT,7817.71,C90045638,53860.00,46042.29,M573487274,0.0,0.00,0
6,1,PAYMENT,7107.77,C154988899,183195.00,176087.23,M408069119,0.0,0.00,0
7,1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.00,0
8,1,PAYMENT,4024.36,C1265012928,2671.00,0.00,M1176932104,0.0,0.00,0
9,1,DEBIT,5337.77,C712410124,41720.00,36382.23,C195600860,41898.0,40348.79,0


In [ ]:
print(df[df['nameDest'].str.startswith('M') & ((df['oldbalanceDest'] != 0) | (df['newbalanceDest'] != 0))].shape) # empty :D
customer_df = df[df['nameDest'].str.startswith('C')]
merchant_df = df[df['nameDest'].str.startswith('M')]

print((merchant_df['isFraud'] == 1).sum())

(0, 10)
0


In [ ]:
senders = set(df['nameOrig'])
receivers = set(df['nameDest'])

chain_accounts = senders.intersection(receivers)
print(f"Accounts that appear as both sender and receiver: {len(chain_accounts)}")

Accounts that appear as both sender and receiver: 1769


In [ ]:
df['errorBalanceOrig'] = df['newbalanceOrig'] - df['oldbalanceOrg'] + df['amount']
df['errorBalanceDest'] = df['oldbalanceDest'] - df['newbalanceDest'] + df['amount']

df.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,errorBalanceOrig,errorBalanceDest
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.0,0.00,0,0.00,9839.64
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.0,0.00,0,0.00,1864.28
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.0,0.00,1,0.00,181.00
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.0,0.00,1,0.00,21363.00
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.0,0.00,0,0.00,11668.14
5,1,PAYMENT,7817.71,C90045638,53860.00,46042.29,M573487274,0.0,0.00,0,0.00,7817.71
6,1,PAYMENT,7107.77,C154988899,183195.00,176087.23,M408069119,0.0,0.00,0,0.00,7107.77
7,1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.00,0,0.00,7861.64
8,1,PAYMENT,4024.36,C1265012928,2671.00,0.00,M1176932104,0.0,0.00,0,1353.36,4024.36
9,1,DEBIT,5337.77,C712410124,41720.00,36382.23,C195600860,41898.0,40348.79,0,0.00,6886.98


In [ ]:
def find_real(df, tolerance):
    return df[(df['errorBalanceOrig'].abs() < tolerance) & (df['errorBalanceDest'].abs() < tolerance)]

intersection = find_real(df, 0.1) #the ones where there is no discrepancy are basically all sent to customers, merchants always have error cause old/new dest balance is zero
print(intersection.shape)

#intx_customer = find_real(customer_df, 0.1) # approx equal to intersection, off by 8
#print(intx_customer.shape)

#intx_merch = df[df['nameDest'].str.startswith('M') & (df['errorBalanceOrig'].abs() < 0.1)]
#print(intx_merch.shape)


(279810, 12)


In [ ]:
print(intersection['isFraud'].value_counts()) # 275k non-fraud, 3.9k fraud

#print(df['isFraud'].value_counts())
#print(intx_customer['isFraud'].value_counts())


isFraud
0    275857
1      3953
Name: count, dtype: int64


In [ ]:
intersection.head(50)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,errorBalanceOrig,errorBalanceDest
1124,1,DEBIT,6765.12,C67620673,50317.00,43551.88,C1122805102,16803.80,23568.91,0,0.000000e+00,1.000000e-02
1169,1,CASH_OUT,9980.32,C1589466857,12936.00,2955.68,C1018298342,0.00,9980.32,0,0.000000e+00,0.000000e+00
1171,1,DEBIT,2763.95,C458817438,60249.00,57485.05,C846947180,10137.14,12901.08,0,0.000000e+00,1.000000e-02
1426,1,DEBIT,3014.72,C1987354705,31568.00,28553.28,C40075281,25407.65,28422.37,0,0.000000e+00,3.637979e-12
1450,1,TRANSFER,40149.38,C428245792,287443.90,247294.52,C1291286504,26426.12,66575.50,0,-5.820766e-11,0.000000e+00
1478,1,DEBIT,5454.05,C470132045,86214.00,80759.95,C1330106945,18590.13,24044.18,0,0.000000e+00,0.000000e+00
1681,1,DEBIT,3548.72,C1596144422,12427.00,8878.28,C564742142,13271.31,16820.03,0,0.000000e+00,0.000000e+00
1746,1,DEBIT,6159.72,C810262298,24815.00,18655.28,C1321640594,12751.13,18910.85,0,0.000000e+00,0.000000e+00
1755,1,DEBIT,1533.55,C2134915053,45093.00,43559.45,C75457651,29936.24,31469.78,0,0.000000e+00,1.000000e-02
1756,1,DEBIT,3256.43,C619241052,10935.00,7678.57,C1292838001,26222.91,29479.34,0,0.000000e+00,0.000000e+00
